#### Custom Vision 모델에 다른 이미지를 적용해본다.

In [2]:
# 추가) API 키 .env 파일에서 불러오기
import os
from dotenv import load_dotenv
load_dotenv()
PREDICTION_KEY = os.getenv("AZURE_CV_KEY")

In [3]:
import requests
import os
import pandas as pd
import time

# PREDICTION_KEY = "key"  #customvision prediction key

ENDPOINT = "https://eastus.api.cognitive.microsoft.com/customvision/v3.0/Prediction/852e9554-f39a-4351-8289-b3b9096b555c/classify/iterations/Iteration2/image"

headers = {
    "Prediction-Key": PREDICTION_KEY,
    "Content-Type": "application/octet-stream"
}

IMAGE_DIR = r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\CustomVision\Classification_predictions"
results = []

for filename in os.listdir(IMAGE_DIR):
    if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    path = os.path.join(IMAGE_DIR, filename)

    with open(path, "rb") as image_data:
        response = requests.post(
            ENDPOINT,
            headers=headers,
            data=image_data
        )

    if response.status_code != 200:
        print("Error:", filename, response.status_code)
        continue

    predictions = response.json()["predictions"]

    p_wide = 0
    p_narrow = 0
    p_barrier_yes = 0
    p_barrier_no = 0

    for pred in predictions:
        tag = pred["tagName"]
        prob = pred["probability"]

        if tag == "wide_lane":
            p_wide = prob
        elif tag == "narrow_lane":
            p_narrow = prob
        elif tag == "barrier_yes":
            p_barrier_yes = prob
        elif tag == "barrier_no":
            p_barrier_no = prob

    results.append({
        "image": filename,
        "p_wide": p_wide,
        "p_narrow": p_narrow,
        "p_barrier_yes": p_barrier_yes,
        "p_barrier_no": p_barrier_no
    })

    time.sleep(0.2)  # 속도 제한 방지

df = pd.DataFrame(results)
df.to_csv(r"C:\Users\EL36\Desktop\1차프로젝트_CCTV\1stProject_MSAI09\CustomVision\0_classification_features.csv", index=False)

print("완료")

FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: 'C:\\Users\\EL36\\Desktop\\1차프로젝트_CCTV\\CustomVision\\Classification_predictions'

#### 결과물 유효여부 확인

In [2]:
import pandas as pd

df = pd.read_csv("classification_features_iteration2.csv")
df.describe()

,p_wide,p_narrow,p_barrier_yes,p_barrier_no
count,3971.000000,3971.000000,3971.000000,3971.000000
mean,0.550530,0.450947,0.405211,0.594749
std,0.355302,0.355473,0.304958,0.304892
min,0.000392,0.001075,0.001396,0.009831
25%,0.183963,0.096434,0.116262,0.317183
50%,0.613217,0.389599,0.356267,0.643954
75%,0.904601,0.818737,0.682423,0.883190
max,0.998933,0.999619,0.990137,0.998610


In [ ]:
df["accident_label"] = df["image"].apply(lambda x: 0 if x.endswith("북쪽.png") else 1)

In [ ]:
#0 = 비사고, 1 = 사고
df.groupby("accident_label")[["p_wide","p_narrow","p_barrier_yes","p_barrier_no"]].mean()

,p_wide,p_narrow,p_barrier_yes,p_barrier_no
accident_label,,,,
0,0.468185,0.533389,0.385920,0.614035
1,0.657856,0.343495,0.430354,0.569612


In [11]:
from scipy.stats import ttest_ind

acc = df[df["accident_label"] == 1]
non = df[df["accident_label"] == 0]

t_wide = ttest_ind(acc["p_wide"], non["p_wide"])
t_barrier = ttest_ind(acc["p_barrier_yes"], non["p_barrier_yes"])

print(t_wide)
print(t_barrier)

print("wide p-value:", f"{t_wide.pvalue:.10f}")
print("barrier p-value:", f"{t_barrier.pvalue:.10f}")

TtestResult(statistic=np.float64(17.2875925812982), pvalue=np.float64(1.2849851913528121e-64), df=np.float64(3969.0))
TtestResult(statistic=np.float64(4.562283137077573), pvalue=np.float64(5.212905521054795e-06), df=np.float64(3969.0))
wide p-value: 0.0000000000
barrier p-value: 0.0000052129


#### 회귀분석

In [13]:
import statsmodels.api as sm

X = df[["p_wide", "p_barrier_yes"]]
X = sm.add_constant(X)
y = df["accident_label"]

model = sm.Logit(y, X).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.618536
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:         accident_label   No. Observations:                 3971
Model:                          Logit   Df Residuals:                     3968
Method:                           MLE   Df Model:                            2
Date:                Mon, 02 Mar 2026   Pseudo R-squ.:                 0.09630
Time:                        10:38:30   Log-Likelihood:                -2456.2
converged:                       True   LL-Null:                       -2717.9
Covariance Type:            nonrobust   LLR p-value:                2.125e-114
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -1.1277      0.067    -16.927      0.000      -1.258      -0.997
p_wide            3.

In [15]:
import numpy as np
np.exp(3.6819), np.exp(-2.9441)

(np.float64(39.72179382839576), np.float64(0.05264942298033528))

In [14]:
from sklearn.linear_model import LogisticRegression

X = df[["p_wide", "p_barrier_yes"]]
y = df["accident_label"]

model = LogisticRegression()
model.fit(X, y)

print("coefficients:", model.coef_)
print("intercept:", model.intercept_)

coefficients: [[ 3.49262233 -2.73280192]]
intercept: [-1.10824743]


### 📊 School Zone Structural Risk Analysis
1. 분석 목적
본 분석의 목적은 다음과 같다:
> 도로 구조적 특성이 어린이 보행 사고다발지 여부에 유의미한 영향을 미치는가?
이를 위해 Custom Vision 모델을 통해 다음 구조 변수들을 확률값 형태로 추출하였다.

* `p_wide` : 도로가 넓을 확률
* `p_barrier_yes` : 물리적 분리장치 존재 확률

#### 2. 단변량 분석 결과 (t-test)
도로 폭 (p_wide)
* t = 17.29
* p < 0.0001
사고다발지는 일반지역보다 유의하게 넓은 도로 특성을 가진다.

#### 물리적 분리장치 (p_barrier_yes)
* t = 4.56
* p < 0.0001
사고다발지에서 분리장치 특성도 통계적으로 차이가 존재한다.
#### 3. 로지스틱 회귀 분석 (구조 변수 2개)
모델
```
accident_label ~ p_wide + p_barrier_yes
```

* 결과 요약

| 변수            | 계수 (coef) | p-value | 해석                 |
| ------------- | --------- | ------- | ------------------ |
| p_wide        | +3.68     | <0.001  | 넓은 도로일수록 사고 확률 증가  |
| p_barrier_yes | -2.94     | <0.001  | 분리장치 존재 시 사고 확률 감소 |

* 오즈비 해석
* exp(3.68) ≈ 39
* exp(-2.94) ≈ 0.05
해석:
* 넓은 도로 특성은 사고 발생 odds를 크게 증가시킨다.
* 물리적 분리장치는 사고 odds를 약 95% 감소시킨다.

#### 4. 모델 설명력
* Pseudo R² ≈ 0.096
구조 변수 2개만으로 약 9.6% 설명력 확보.


In [ ]:
from scipy.stats import ttest_ind

acc = df[df["accident_label"] == 1]
non = df[df["accident_label"] == 0]

t_wide = ttest_ind(acc["p_wide"], non["p_wide"])
t_barrier = ttest_ind(acc["p_barrier_yes"], non["p_barrier_yes"])

print(t_wide)
print(t_barrier)

print("wide p-value:", f"{t_wide.pvalue:.10f}")
print("barrier p-value:", f"{t_barrier.pvalue:.10f}")

TtestResult(statistic=np.float64(17.2875925812982), pvalue=np.float64(1.2849851913528121e-64), df=np.float64(3969.0))
TtestResult(statistic=np.float64(4.562283137077573), pvalue=np.float64(5.212905521054795e-06), df=np.float64(3969.0))
wide p-value: 0.0000000000
barrier p-value: 0.0000052129


### 확장 분석 (추가 변수 포함)

이제 다음 변수들을 추가하여 다변량 분석을 수행한다

* `parked_density`
* `sidewalk_ratio`
* `road_width_relative`
